# Deployment — Cloudflare Quick Tunnel Public Demo

## 1. Deployment Overview

The VIGILOX research and production work culminated in a public browser-accessible deployment using **Cloudflare Quick Tunnel**.

The objective of this deployment was not to create a permanent cloud infrastructure stack.

The goal was to verify that the complete VIGILOX application could be accessed over public HTTPS and that the main document workflow continued to work outside the local browser environment.

The deployed path was:

```text
Internet
   ↓
Cloudflare Quick Tunnel
   ↓
Local FastAPI Application
   ↓
PostgreSQL
   ↑
Background Worker
   ↓
PaddleOCR + Groq
```

This deployment was used for:

- public HTTPS verification
- browser acceptance testing
- end-to-end document processing
- original source rendering
- evidence overlay validation
- human review testing
- final-state persistence verification

---

# 2. Deployment Objective

The deployment objective was:

> Expose the locally running VIGILOX application through a secure public HTTPS tunnel without changing the existing FastAPI, worker, OCR, LLM, or PostgreSQL architecture.

The existing local application already consisted of:

```text
FastAPI API
+
PostgreSQL
+
Background Worker
+
PaddleOCR
+
Groq
```

Cloudflare was added only as the public network path:

```text
Public Browser
      ↓
Cloudflare
      ↓
localhost:8000
```

---

# 3. Why Cloudflare Quick Tunnel Was Used

The application needed a simple public deployment path for demonstration and verification.

Cloudflare Quick Tunnel was suitable because it allowed the local FastAPI application to be exposed through a generated HTTPS endpoint.

The approach avoided changing the internal VIGILOX architecture.

The application continued to run exactly as it did locally:

```text
FastAPI
    ↓
PostgreSQL
    ↑
Worker
```

The tunnel acted only as the public ingress layer.

---

# 4. Deployment Architecture

The final public demo architecture was:

```text
                 Public Browser
                       │
                       ▼
              Cloudflare Network
                       │
                       ▼
             Cloudflare Quick Tunnel
                       │
                       ▼
              127.0.0.1:8000
                       │
                       ▼
                 FastAPI API
                       │
              ┌────────┴────────┐
              │                 │
              ▼                 ▼
         PostgreSQL       Pending Storage
              │
              ▼
            Worker
              │
       ┌──────┴──────┐
       │             │
       ▼             ▼
   PaddleOCR      Groq API
       │             │
       └──────┬──────┘
              ▼
        Validation
              ↓
        Persistence
              ↓
        Human Review
```

---

# 5. Deployment Prerequisites

Before starting the public tunnel, the following components must already be available locally:

```text
Python Environment
PostgreSQL
VIGILOX Database
FastAPI Application
Background Worker
Groq API Configuration
PaddleOCR Runtime
cloudflared
```

The application environment also requires a valid:

```env
DATABASE_URL
GROQ_API_KEY
```

The default extraction model used by the project is:

```env
VIGILOX_GROQ_MODEL=openai/gpt-oss-20b
```

---

# 6. Verify PostgreSQL

PostgreSQL must be running before starting the API or worker.

The VIGILOX application uses PostgreSQL for:

- documents
- analysis
- jobs
- batches
- reviews
- audit history
- worker heartbeats

The database schema should already be migrated.

Typical migration command:

```powershell
python -m alembic upgrade head
```

Schema state can be checked with:

```powershell
python -m alembic current
```

---

# 7. Start the FastAPI Application

Open the first PowerShell terminal.

Navigate to the VIGILOX project:

```powershell
cd C:\path\to\VIGILOX-Document-Intelligence
```

Activate the virtual environment:

```powershell
.\.venv\Scripts\Activate.ps1
```

Start FastAPI:

```powershell
python -m uvicorn backend.app.main:app --host 127.0.0.1 --port 8000
```

The application is now available locally at:

```text
http://127.0.0.1:8000
```

---

# 8. Verify API Health

Before starting Cloudflare, verify the local API.

Run:

```powershell
Invoke-WebRequest http://127.0.0.1:8000/health -UseBasicParsing
```

Expected result:

```text
StatusCode : 200
```

Example response:

```json
{
  "status": "ok",
  "service": "vigilox-document-intelligence",
  "version": "0.1.0"
}
```

This confirms that Cloudflare will have a valid local origin to forward requests to.

---

# 9. Start the Background Worker

Open a second PowerShell terminal.

Navigate to the project:

```powershell
cd C:\path\to\VIGILOX-Document-Intelligence
```

Activate the environment:

```powershell
.\.venv\Scripts\Activate.ps1
```

Start the worker:

```powershell
python -m backend.worker
```

The worker is responsible for:

```text
Claiming Durable Jobs
      ↓
PaddleOCR
      ↓
Groq Structured Extraction
      ↓
Evidence Validation
      ↓
Quality Checks
      ↓
Persistence
```

Without the worker, the browser can create jobs, but asynchronous document processing cannot complete.

---

# 10. Cloudflared Installation

The Windows 64-bit Cloudflare executable was used.

The executable was named:

```text
cloudflared-windows-amd64.exe
```

The executable can be kept in a local tools or Downloads directory.

No modification to the VIGILOX Python environment is required.

---

# 11. Verify Cloudflared

Navigate to the folder containing the executable.

Example:

```powershell
cd $env:USERPROFILE\Downloads
```

Verify that the executable exists:

```powershell
Get-ChildItem cloudflared*.exe
```

Then check the installed version:

```powershell
.\cloudflared-windows-amd64.exe --version
```

During the verified deployment, the executable reported:

```text
cloudflared version 2026.8.2
```

This confirmed that the binary was runnable before the tunnel was started.

---

# 12. Start the Cloudflare Tunnel

Open a third PowerShell terminal.

Navigate to the directory containing `cloudflared`:

```powershell
cd $env:USERPROFILE\Downloads
```

Start the tunnel:

```powershell
.\cloudflared-windows-amd64.exe tunnel `
  --protocol http2 `
  --url http://127.0.0.1:8000
```

The tunnel connects the public Cloudflare network to the local FastAPI service.

---

# 13. Generated Public URL

When the tunnel is created successfully, Cloudflare prints a generated HTTPS address.

Example format:

```text
https://generated-name.trycloudflare.com
```

The exact generated address is intentionally not stored in the research notes because a new tunnel session can produce a different URL.

The generated origin can then be used with the normal VIGILOX routes.

For example:

```text
https://generated-name.trycloudflare.com/dashboard
https://generated-name.trycloudflare.com/upload
https://generated-name.trycloudflare.com/documents
https://generated-name.trycloudflare.com/review
```

---

# 14. Initial Tunnel Timeout

During the first deployment attempt, `cloudflared` returned:

```text
failed to request quick Tunnel:
context deadline exceeded
```

The failure occurred while requesting a tunnel from:

```text
https://api.trycloudflare.com/tunnel
```

This did not indicate a VIGILOX API failure.

The local application health endpoint was still returning:

```text
HTTP 200
```

---

# 15. Network Diagnostics

Connectivity was checked using PowerShell.

Cloudflare API:

```powershell
Test-NetConnection api.trycloudflare.com -Port 443
```

Cloudflare tunnel edge:

```powershell
Test-NetConnection region1.v2.argotunnel.com -Port 7844
```

The local VIGILOX API was also checked:

```powershell
Invoke-WebRequest http://127.0.0.1:8000/health -UseBasicParsing
```

The tests showed that:

```text
Cloudflare API reachable
Cloudflare tunnel edge reachable
Local VIGILOX API healthy
```

---

# 16. DNS Refresh

The Windows DNS resolver cache was flushed:

```powershell
ipconfig /flushdns
```

The tunnel command was then executed again.

This time the tunnel was successfully created.

---

# 17. HTTP/2 Tunnel Protocol

The deployment explicitly used:

```text
HTTP/2
```

through:

```powershell
--protocol http2
```

The successful tunnel connection reported:

```text
protocol=http2
```

The Cloudflare connectivity pre-check also verified:

```text
DNS Resolution
UDP Connectivity
TCP Connectivity
Cloudflare API
```

---

# 18. Successful Tunnel Registration

The successful Cloudflare output included a registered tunnel connection.

Conceptually:

```text
Quick Tunnel Created
       ↓
Connector Generated
       ↓
Cloudflare Edge Connection
       ↓
HTTP/2 Registered
```

The tunnel was connected through a Cloudflare edge location and became accessible over public HTTPS.

---

# 19. Public Browser Verification

Once the tunnel was active, the VIGILOX application was opened through the generated public URL.

The following pages were verified:

```text
/dashboard
/upload
/documents
/review
/review/{document_id}
```

All major pages loaded through the public Cloudflare route.

---

# 20. Dashboard Verification

The public Dashboard successfully displayed operational information from PostgreSQL.

This verified the chain:

```text
Public Browser
      ↓
Cloudflare
      ↓
FastAPI
      ↓
Dashboard Service
      ↓
PostgreSQL
      ↓
Browser UI
```

The page also showed application availability through the public route.

---

# 21. Upload Verification

A document was submitted through the public `/upload` page.

The request traveled through:

```text
Browser
  ↓
Cloudflare
  ↓
FastAPI
  ↓
Job Creation
```

The upload successfully created a durable processing job.

---

# 22. Worker Processing Verification

After public upload:

```text
Job
 ↓
PostgreSQL Queue
 ↓
Local Worker
```

The worker successfully claimed the job and performed:

```text
OCR
Structured Extraction
Validation
Persistence
```

This proved that using a public tunnel did not change the internal worker architecture.

---

# 23. OCR Verification

The uploaded document successfully reached PaddleOCR through the normal worker pipeline.

The OCR output was used by the structured extraction service.

This verified:

```text
Public Upload
      ↓
Local Worker
      ↓
PaddleOCR
```

through the deployed public workflow.

---

# 24. Groq Extraction Verification

After OCR, the structured extraction layer used Groq as normal.

Conceptually:

```text
OCR Text
   ↓
Groq
   ↓
Structured Fields
```

The public deployment did not require a separate LLM configuration.

The worker continued using the same configured provider settings.

---

# 25. PostgreSQL Persistence Verification

The completed processing result was persisted to PostgreSQL.

The processed document then appeared in:

```text
Documents
Review Queue
Document Workspace
```

through the public interface.

This confirmed that:

```text
Public Access
```

was only a network change.

The existing PostgreSQL system of record remained unchanged.

---

# 26. Document Workspace Verification

A completed document was opened through the public document workspace.

The workspace successfully displayed:

- document metadata
- extracted fields
- confidence information
- validation findings
- image quality
- final-state information
- reviewer controls

This verified the full API-to-frontend path through Cloudflare.

---

# 27. Original Source Image Verification

The original uploaded source document was rendered through the public route.

The browser requested:

```text
/api/v1/documents/{document_id}/image
```

through Cloudflare.

The original image displayed correctly in the review workspace.

This verified:

```text
Managed Storage
      ↓
FastAPI Image Endpoint
      ↓
Cloudflare
      ↓
Browser
```

---

# 28. Evidence Overlay Verification

The public document workspace also displayed OCR evidence overlays.

The chain was:

```text
OCR Bounding Boxes
       ↓
Persisted Evidence
       ↓
API Response
       ↓
Browser Scaling
       ↓
Source Image Overlay
```

The evidence regions aligned with the original source document.

---

# 29. Human Review Verification

A human review action was performed through the public deployment.

The workflow was:

```text
Open Document
      ↓
Inspect Source
      ↓
Inspect Extracted Fields
      ↓
Inspect Evidence
      ↓
Submit Review Action
      ↓
Backend Authorization
      ↓
Database Persistence
```

The resulting final state was successfully stored.

---

# 30. Audit Verification

The review workspace displayed persisted reviewer information after the human review action.

This verified that the public request reached:

```text
Review API
   ↓
Review Service
   ↓
PostgreSQL
   ↓
Audit / Review Record
```

The audit workflow therefore remained functional through the Cloudflare route.

---

# 31. Complete Public Workflow

The verified deployment flow was:

```text
Public HTTPS
      ↓
Dashboard
      ↓
Upload
      ↓
Durable Job
      ↓
Worker
      ↓
PaddleOCR
      ↓
Groq
      ↓
Validation
      ↓
PostgreSQL
      ↓
Documents
      ↓
Review Workspace
      ↓
Original Source
      ↓
Evidence Overlay
      ↓
Human Review
      ↓
Final Record
      ↓
Audit History
```

This was the primary success criterion of the deployment.

---

# 32. Three-Terminal Runtime Model

The verified deployment used three PowerShell terminals.

## Terminal 1

```text
FastAPI
```

Command:

```powershell
python -m uvicorn backend.app.main:app --host 127.0.0.1 --port 8000
```

---

## Terminal 2

```text
Worker
```

Command:

```powershell
python -m backend.worker
```

---

## Terminal 3

```text
Cloudflare Tunnel
```

Command:

```powershell
.\cloudflared-windows-amd64.exe tunnel `
  --protocol http2 `
  --url http://127.0.0.1:8000
```

PostgreSQL remained available as the database service.

---

# 33. Runtime Dependency Relationship

The deployed public application depends on the local services remaining available.

Conceptually:

```text
Cloudflare Tunnel
      ↓
FastAPI
      ↓
PostgreSQL
      ↑
Worker
```

If the API stops, public HTTP requests cannot be served.

If the worker stops, asynchronous documents remain waiting.

If PostgreSQL stops, the application loses access to its system of record.

If the tunnel stops, the local application remains available locally but is no longer exposed through the generated public route.

---

# 34. Security Scope of the Demo

The deployment used the existing local reviewer identity configuration.

Example:

```env
VIGILOX_REVIEW_IDENTITY_MODE=local_env
```

This made the deployment suitable for a controlled demonstration and verification workflow.

It was not used as a replacement for the production trusted-proxy identity architecture implemented during Phase 11.

---

# 35. Why Local Reviewer Mode Matters

With local reviewer mode:

```text
Server Configuration
      ↓
Reviewer Identity
```

rather than:

```text
Authenticated Enterprise User
      ↓
Identity Provider
```

Therefore the public tunnel was treated as a controlled demo environment.

The deployment did not claim enterprise authentication.

---

# 36. Cloudflare and the Application Security Model

Cloudflare Quick Tunnel provided public HTTPS connectivity.

It did not replace:

- backend authorization
- reviewer identity logic
- request validation
- storage protections
- database constraints
- API error handling

Those remained VIGILOX responsibilities.

The tunnel only changed how the browser reached the application.

---

# 37. No Application Code Change Required

One of the useful deployment findings was that no major application redesign was required.

The same local application:

```text
127.0.0.1:8000
```

could be exposed through Cloudflare.

This validated the benefit of keeping the application HTTP interface self-contained and same-origin.

---

# 38. Same-Origin Frontend Behavior

The frontend continued using relative API routes.

For example:

```text
/api/v1/documents
/api/v1/document-jobs
/api/v1/dashboard
```

Because the frontend and API were served by the same FastAPI origin, the browser automatically sent those requests through the Cloudflare hostname.

No separate frontend API base URL was required.

---

# 39. CORS Behavior

The same-origin architecture also meant the public tunnel did not require a broad CORS configuration.

Conceptually:

```text
Page:
https://generated-name.trycloudflare.com

API:
https://generated-name.trycloudflare.com/api/...
```

Both share the same origin.

This maintained the existing security model.

---

# 40. Source Image Same-Origin Behavior

The original source image also remained same-origin:

```text
https://generated-name.trycloudflare.com/api/v1/documents/{id}/image
```

This was important for both:

- CSP compatibility
- evidence-overlay rendering

---

# 41. Deployment Verification vs Production Infrastructure

The Cloudflare deployment should be understood as:

```text
Public Application Verification
```

The production-oriented infrastructure designed in Phase 11 remains:

```text
Nginx
  ↓
FastAPI
  ↓
PostgreSQL
  ↑
Worker
```

with Docker/Compose configuration and trusted identity boundaries.

The two approaches serve different purposes.

---

# 42. Cloudflare Demo Architecture

```text
Cloudflare
    ↓
Local API
```

was used to verify public accessibility.

---

# 43. Production-Oriented Architecture

The production design remains:

```text
Public Traffic
      ↓
Nginx
      ↓
FastAPI
      ↓
PostgreSQL
      ↑
Worker
```

with:

```text
Managed Storage
Pending Storage
Migrations
Monitoring
Security Boundary
```

The research notes preserve this distinction.

---

# 44. Deployment Troubleshooting Workflow

When the tunnel did not start initially, troubleshooting followed a layered approach.

```text
Is VIGILOX healthy locally?
      ↓
Is Cloudflare API reachable?
      ↓
Is tunnel edge reachable?
      ↓
Is DNS resolution working?
      ↓
Retry tunnel
```

This avoided incorrectly changing application code for a network connectivity problem.

---

# 45. Local Health First

Before investigating Cloudflare:

```powershell
Invoke-WebRequest http://127.0.0.1:8000/health -UseBasicParsing
```

confirmed:

```text
StatusCode : 200
```

This isolated the problem away from FastAPI.

---

# 46. Cloudflare API Test

Cloudflare API connectivity was tested with:

```powershell
Test-NetConnection api.trycloudflare.com -Port 443
```

This confirmed that HTTPS access to the Cloudflare service was possible.

---

# 47. Tunnel Edge Test

The tunnel edge was tested using:

```powershell
Test-NetConnection region1.v2.argotunnel.com -Port 7844
```

This confirmed access to the Cloudflare tunnel network.

---

# 48. Retry After DNS Flush

After:

```powershell
ipconfig /flushdns
```

the same tunnel command succeeded.

This showed that deployment troubleshooting should verify the network path before modifying application configuration.

---

# 49. Deployment Validation Result

The public demo successfully verified:

```text
FastAPI Public Access
Dashboard
Upload
Durable Jobs
Worker Processing
PaddleOCR
Groq Extraction
PostgreSQL Persistence
Documents
Review Queue
Document Workspace
Original Source Image
Evidence Overlay
Human Review
Final State
Audit Information
```

This represented a full end-to-end deployment smoke test.

---

# 50. What Was Not Required for the Demo

The Quick Tunnel verification did not require changes to:

```text
OCR Pipeline
Groq Extraction
PostgreSQL Models
Worker Queue
Document Review Logic
Frontend Routes
```

The deployment reused the existing system.

This is a positive architecture result because deployment exposure remained separate from core document-processing behavior.

---

# 51. Research Value of the Deployment

The deployment demonstrated several important engineering properties.

## 51.1 Network Transparency

The application behaved the same whether reached through:

```text
localhost
```

or:

```text
Cloudflare HTTPS endpoint
```

---

## 51.2 Durable Processing Independence

The worker did not care whether the job originated from a local browser or a public browser.

Once the job reached PostgreSQL:

```text
Browser Location
```

was irrelevant to processing.

---

## 51.3 Same-Origin Design Simplified Public Access

Because:

```text
Frontend
API
Document Images
```

were all served by the same application, exposing one local HTTP origin was enough to expose the complete system.

---

## 51.4 Backend State Remained Authoritative

The tunnel introduced no new business state.

The authoritative state remained:

```text
PostgreSQL
```

---

# 52. Lessons Learned

## 52.1 Deployment Problems Are Not Always Application Problems

The first tunnel timeout occurred while the local VIGILOX API was healthy.

The correct troubleshooting process isolated:

```text
Application
Network
Cloudflare
```

before making changes.

---

## 52.2 Public Exposure Should Not Change Core Architecture

A good deployment method should expose the application without forcing business logic changes.

The Quick Tunnel satisfied this goal.

---

## 52.3 Same-Origin Applications Are Easier to Expose Safely

A single public origin simplified:

- API routing
- image routing
- browser security
- CSP
- evidence overlays

---

## 52.4 Background Workers Make Deployment More Reliable

The public browser did not need to remain connected while processing completed.

The durable job system from Phase 9 worked unchanged through the tunnel.

---

## 52.5 Public Browser Verification Adds Confidence Beyond Local Testing

Testing through an external HTTPS route verified:

- URL routing
- browser asset loading
- API requests
- document image delivery
- review operations

under a more realistic access path.

---

# 53. Deployment Deliverables

The deployment phase completed:

- `cloudflared` Windows executable verification
- local API health verification
- Cloudflare connectivity checks
- HTTP/2 tunnel startup
- public HTTPS access
- Dashboard verification
- Upload verification
- async job verification
- worker processing verification
- PaddleOCR verification
- Groq extraction verification
- PostgreSQL persistence verification
- document library verification
- review queue verification
- document workspace verification
- original image verification
- evidence overlay verification
- human review verification
- final-state persistence verification
- audit information verification

---

# 54. Final Deployment Procedure

The complete procedure can be summarized as follows.

## Step 1 — Start PostgreSQL

Ensure the configured VIGILOX PostgreSQL database is available.

---

## Step 2 — Start API

```powershell
cd C:\path\to\VIGILOX-Document-Intelligence
.\.venv\Scripts\Activate.ps1

python -m uvicorn backend.app.main:app --host 127.0.0.1 --port 8000
```

---

## Step 3 — Verify API

```powershell
Invoke-WebRequest http://127.0.0.1:8000/health -UseBasicParsing
```

---

## Step 4 — Start Worker

```powershell
cd C:\path\to\VIGILOX-Document-Intelligence
.\.venv\Scripts\Activate.ps1

python -m backend.worker
```

---

## Step 5 — Start Cloudflare

```powershell
cd $env:USERPROFILE\Downloads

.\cloudflared-windows-amd64.exe tunnel `
  --protocol http2 `
  --url http://127.0.0.1:8000
```

---

## Step 6 — Open Generated URL

Cloudflare returns:

```text
https://generated-name.trycloudflare.com
```

Open:

```text
/generated root
/dashboard
/upload
/documents
/review
```

through that generated origin.

---

## Step 7 — Run End-to-End Smoke Test

```text
Upload Test Document
      ↓
Wait for Worker
      ↓
Open Document
      ↓
Inspect Source
      ↓
Inspect Evidence
      ↓
Complete Review
      ↓
Confirm Final State
```

---

# 55. Final Deployment Architecture

```text
                         Internet
                            │
                            ▼
                  ┌──────────────────┐
                  │    Cloudflare    │
                  │   Quick Tunnel   │
                  └────────┬─────────┘
                           │
                           ▼
                  ┌──────────────────┐
                  │     FastAPI      │
                  │  127.0.0.1:8000 │
                  └────────┬─────────┘
                           │
               ┌───────────┴───────────┐
               │                       │
               ▼                       ▼
       ┌───────────────┐       ┌───────────────┐
       │  PostgreSQL   │       │ Pending Files │
       └───────┬───────┘       └───────────────┘
               │
               ▼
       ┌───────────────┐
       │    Worker     │
       ├───────────────┤
       │ PaddleOCR     │
       │ Groq          │
       │ Validation    │
       │ Persistence   │
       └───────┬───────┘
               │
               ▼
       ┌───────────────┐
       │ Final Record  │
       │ Review Audit  │
       └───────────────┘
```

---

# 56. Final Outcome

The deployment successfully demonstrated that the complete VIGILOX workflow could operate through a public HTTPS entry point without changing the core application architecture.

The final verified path was:

```text
Public Browser
      ↓
Cloudflare
      ↓
FastAPI
      ↓
Durable PostgreSQL Job
      ↓
Worker
      ↓
PaddleOCR
      ↓
Groq
      ↓
Validation
      ↓
Persistence
      ↓
Review Workspace
      ↓
Human Review
      ↓
Final Record
```

This confirmed that the system worked beyond localhost-only browser access.

The deployment also validated the architectural separation established throughout the project:

```text
Network Exposure
        ≠
Business Logic

Browser
        ≠
Job Queue

FastAPI
        ≠
OCR Worker

Machine Extraction
        ≠
Final Authority
```

The Cloudflare deployment therefore served as the final public end-to-end demonstration of the VIGILOX Document Intelligence system.

---

## Deployment Summary

| Area | Result |
|---|---|
| Local FastAPI Health | Verified |
| Cloudflared Installation | Verified |
| Cloudflare Connectivity | Verified |
| HTTP/2 Tunnel | Established |
| Public HTTPS Access | Verified |
| Dashboard | Verified |
| Upload | Verified |
| Durable Job Creation | Verified |
| Background Worker | Verified |
| PaddleOCR | Verified |
| Groq Extraction | Verified |
| PostgreSQL Persistence | Verified |
| Documents Page | Verified |
| Review Queue | Verified |
| Document Workspace | Verified |
| Original Source Image | Verified |
| OCR Evidence Overlay | Verified |
| Human Review | Verified |
| Final-State Persistence | Verified |
| Audit Information | Verified |

---

**VIGILOX Research Development Sequence**

```text
Phase 1–7C
Core OCR, Extraction, Validation and Review Research
        ↓
Phase 8
Professional UI / UX
        ↓
Phase 9
Async Processing, Durable Jobs and Performance
        ↓
Phase 10
Advanced Document Intelligence
        ↓
Phase 11
Production Hardening and Security
        ↓
Phase 12
Final Verification and Release Readiness
        ↓
Deployment
Public Cloudflare End-to-End Verification
```